# Python for NLP: Tokenisation, Small Models, and Text Pipelines

## Lab Two (Estimated Duration: ~90 minutes)

This lab follows the root `README.md` syllabus: tokenisation, small-model inference, classification, summarisation, information extraction, validation, and failure modes.

Everything uses small local examples. The aim is to understand the workflow, not to build a perfect NLP system.

**LLM orientation:** this lab uses simple Python and small local models because they make the steps visible. Large language models use more complex versions of the same broad pattern: text is split into tokens, converted into numbers, processed by a model, and checked against a task. We will not cover transformer architecture, embeddings, or fine-tuning in detail here, but the lab gives vocabulary for understanding those topics later.

**Useful LLM terms for this lab:**
- A `token` is a model-readable piece of text. It may be a word, part of a word, punctuation, or another small unit.
- An `embedding` is a numerical representation of text. Unlike the word counts in this lab, embeddings are designed to capture similarity and meaning, not just exact word frequency.
- A `prompt` is the instruction and context given to an LLM. Prompting can sometimes replace a small classifier, but the output still needs checking.
- A `context window` is the amount of text an LLM can consider at once. Longer documents may need chunking, summarising, or retrieval.
- `Hallucination` means the model produces something fluent but unsupported or false. This is why validation and human review matter.
- `RAG`, or retrieval-augmented generation, means giving the model relevant source material before asking it to answer or summarise.

Rules:
- Print intermediate outputs.
- Do not use API calls or downloaded LLMs.
- Treat model outputs as things to check.

## How to Use This Notebook

- This lab notebook is self-contained; it does not import from or depend on the solution notebook.
- Run cells from top to bottom.
- Read the markdown before each code cell; it tells you what the code is trying to do.
- Use the `Programming Plan` before typing. It breaks the task into small steps.
- If you get an error, read the last line first, then check whether an earlier cell was skipped.
- Keep outputs small and inspect them often with `print()`.
- Do not use confidential health, social care, interview, or institutional data in these exercises.
- The matching solution notebook has the same structure and can be used after you try the lab.


## Part A: Texts and Tokenisation (Approx. 25 minutes)

**Why This Matters**

Tokenisation is the first modelling decision in many NLP systems. A tokeniser decides what counts as a unit: a word, a number, punctuation, or part of a domain-specific phrase. Different tokenisation choices can change what a model can see.

**LLM connection:** LLMs also tokenize text, but often into subword pieces rather than whole words. For example, an unfamiliar word may be split into smaller fragments. The details are different from `.split()` or our regex tokeniser, but the idea is the same: the model cannot work with raw text until the text has been turned into model-readable pieces.

### Question 1: Word Count with `split`

Count the words in `note`.

Guidance:
- make `tokens = note.split()`
- count with `len(tokens)`
- print the count

**Programming Plan**

- Call `.split()` on the text to get tokens.
- Store the tokens in a variable before counting.
- Use `len(tokens)` for the count.
- Print both tokens and count if you are unsure.

In [1]:
note = "Public access survey mentions missed appointments because bus service was cancelled"

In [2]:
# Answer 1
# Starter code that runs.
# Replace the next two lines with your own use of note.split() and len(...).
tokens = []
word_count = 0

print(f"Tokens: {tokens}")
print(f"Word count: {word_count}")

Tokens: []
Word count: 0


### Question 2: Clean and Split Text

Clean `message`, then split it into tokens.

Guidance:
- lower case the text
- remove `:` and `!`
- print the cleaned text and tokens

**Programming Plan**

- Create a new variable for the cleaned text instead of overwriting immediately.
- Apply `.lower()` first.
- Use `.replace(...)` for each punctuation mark.
- Split only after cleaning.

In [3]:
message = "URGENT: Resident cannot afford travel to clinic!"

In [4]:
# Answer 2

### Question 3: A Small Regex Tokeniser

Use `re.findall(token_pattern, clinical_text)`.

Guidance:
- run the tokeniser
- print the token list
- inspect `7.8%` and `follow-up`

**Programming Plan**

- Importing `re` has already been done for you.
- Use `re.findall(pattern, text)`.
- Store the result in a token variable.
- Inspect the printed list rather than assuming it is correct.

**Before You Code**

Regular expressions can look intimidating, but here you only need to use the provided pattern. The important question is interpretive: does the resulting token list preserve the parts of the text that matter for the research task?

In [5]:
import re

clinical_text = "HbA1c is 7.8%, follow-up in 2 weeks."
token_pattern = r"[A-Za-z0-9]+(?:[-.][A-Za-z0-9]+)*%?|[^\w\s]"
stricter_token_pattern = (
    r"[A-Za-z]+[A-Za-z0-9]*(?:-[A-Za-z0-9]+)*"
    r"|\d+(?:\.\d+)?%?[A-Za-z]*"
)

In [6]:
# Answer 3

### Question 4: Compare Tokenisation Across Text Types

Run the same regex tokeniser on four short examples.

Question 4 is not asking you to invent a perfect tokeniser. It asks you to run the same pattern on different kinds of text and inspect what it keeps, what it splits, and what it treats as a separate token.

The current pattern is:
`[A-Za-z0-9]+(?:[-.][A-Za-z0-9]+)*%?|[^\w\s]`

Read it in two halves, separated by `|`, which means "or":
- `[A-Za-z0-9]+` matches one or more letters or digits.
- `(?:[-.][A-Za-z0-9]+)*` allows extra chunks that start with `-` or `.`, so examples like `follow-up`, `NHS-111`, and `7.8` can stay together.
- `%?` allows an optional percent sign at the end, so `7.8%` can stay as one token.
- `[^\w\s]` is the fallback: it matches one character that is not a word character and not whitespace. This is why punctuation such as `,`, `.`, and `:` appears as separate tokens.

This is deliberately quite broad. It preserves punctuation, and it treats many letter/number strings joined by hyphens or full stops as valid tokens. That can be useful for inspection, but it can also keep noise.

A slightly more restrictive example is stored as `stricter_token_pattern`:
`[A-Za-z]+[A-Za-z0-9]*(?:-[A-Za-z0-9]+)*|\d+(?:\.\d+)?%?[A-Za-z]*`

This stricter version keeps words, simple domain codes such as `HbA1c` and `NHS-111`, hyphenated words such as `follow-up`, and numbers such as `12.5%` or `48h`. It drops standalone punctuation because punctuation is not included as a fallback token. It is still only an example: it will not handle every language, apostrophe, currency symbol, or specialist notation correctly.

Guidance:
- define `regex_tokenise(text)`
- loop through `tokenisation_examples.items()`
- print the example name and tokens

**Programming Plan**

- Put the regex call inside a small function.
- Loop through `.items()` so you have both the example name and text.
- Print a blank line between examples for readability.
- Compare where the tokeniser does well and where it looks fragile.
- Optional: try `stricter_token_pattern` in the same function and note what changes.

In [7]:
tokenisation_examples = {
    "english": "The public service user missed two appointments this month.",
    "non_english": "El paciente no asistio a la cita por ansiedad.",
    "numerical": "Income fell by 12.5% after rent rose by 80 pounds.",
    "domain_specific": "NHS-111 referral: COPD follow-up needed within 48h.",
}

In [8]:
# Answer 4

## Part B: Classifying Short Texts with a Small Local Model (Approx. 35 minutes)

**Working Pattern**

The classifier in this section is intentionally small. The workflow is what matters: labelled examples, feature extraction, model fitting, prediction, and comparison against human labels. This same workflow applies when the model is much larger.

**LLM connection:** with an LLM, you might classify text by writing a prompt instead of training a classifier from scratch. You might also use embeddings, fine-tuning, or a smaller model wrapped around LLM outputs. Those approaches are more advanced than this lab, but they still depend on the same questions: what are the examples, what are the labels, what counts as evidence, and how will we check errors?

### Question 5: Build a Tiny Labelled Dataset

Create a DataFrame from `records`.

**LLM connection:** even when using an LLM through prompts, you still need examples like these. A small labelled table can become a test set, a few-shot prompt, or a reference set for checking whether the model is doing the task you intended.

Guidance:
- import `pandas`
- create `notes_df`
- print the label counts

**Programming Plan**

- Turn the list of dictionaries into a table object.
- Check the first few rows before modelling.
- Count labels so you know whether the classes are balanced.
- Keep the text column and label column names consistent.

In [9]:
records = [
    {"text": "Public access survey mentions missed clinic because bus route was cancelled", "label": "access_barrier"},
    {"text": "Community respondent cannot afford taxi to hospital appointment", "label": "access_barrier"},
    {"text": "Rent increase caused anxiety and arrears", "label": "housing_stress"},
    {"text": "Family reports eviction notice and rent arrears", "label": "housing_stress"},
    {"text": "Public advice note reports chest pain and need for clinical advice", "label": "health_need"},
    {"text": "Public advice note mentions medication side effects causing dizziness", "label": "health_need"},
]

In [10]:
# Answer 5

### Question 6: Train a Small Local Classifier

Fit a `CountVectorizer` + `MultinomialNB` pipeline.

These three imports set up the two main jobs in a basic text classifier:
- `CountVectorizer` converts raw strings into a table of word/token counts. Machine-learning models need numbers, so this is the feature-extraction step: each row is a document, and each column is a word or token from the training data.
- `MultinomialNB` is a Naive Bayes classifier that works well with count data. It learns which words are more common in each label, then uses those word counts to predict the most likely label for new text.
- `Pipeline` joins the vectorizer and classifier into one object. When you call `.fit(...)`, it first learns the vocabulary and count features, then fits the classifier. When you call `.predict(...)`, it applies the same vectorization steps before predicting labels.

The named steps below are `vectorizer` and `classifier`, so you can read the model as: turn text into counts, then classify those counts.

**Mathematical intuition for `MultinomialNB`:** after `CountVectorizer`, each note becomes a count vector $x = (x_1, x_2, \ldots, x_V)$, where $x_j$ is the count of vocabulary token $j$ and $V$ is the vocabulary size.

For each possible label $c$, Naive Bayes estimates two kinds of probabilities:
- $P(c)$: how common label $c$ is in the training data.
- $P(w_j \mid c)$: how common token $w_j$ is among training examples with label $c$.

The model then gives each label a score like this:
$$\log P(c \mid x) = \log P(c) + \sum_{j=1}^{V} x_j \log P(w_j \mid c) + \text{constant}$$

The prediction is the label with the largest score. A token that appears many times has a larger effect because its log-probability is multiplied by its count $x_j$.

The word `multinomial` means the model is designed for counts across many possible token categories. The word `naive` means it treats token counts as if they provide separate pieces of evidence once the label is known. That assumption is not literally true for language, because word order and grammar matter, but it often works surprisingly well for short text classification.

In practice, scikit-learn also uses smoothing, controlled by `alpha`, so an unseen token does not get probability zero. A simplified smoothing formula is:
$$P(w_j \mid c) = \frac{\text{count}(w_j, c) + \alpha}{\sum_k \text{count}(w_k, c) + \alpha V}$$

**LLM connection:** this count-based representation is much simpler than an embedding. Count vectors mostly know which words appear. Embeddings try to place related texts near each other numerically, even when they use different words. Both are ways of turning text into numbers before modelling.

Guidance:
- run the pipeline cell
- fit on `notes_df["text"]` and `notes_df["label"]`
- print `"Model fitted"`

**Programming Plan**

- Fit the pipeline only after `notes_df` exists.
- The input `X` is the text column.
- The target `y` is the label column.
- Print a short success message after `.fit(...)`.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

text_model = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", MultinomialNB()),
])

In [12]:
# Answer 6

### Question 7: Predict and Compare with Human Labels

Use the model on three validation examples.

Guidance:
- create `validation_df`
- add `model_label`
- add `correct`
- print the comparison table

**Programming Plan**

- Create a validation table from the validation records.
- Predict labels for the validation text.
- Compare predicted labels with human labels using `==`.
- Print only the columns needed for inspection.

**Validation Note**

A model label is not a finding by itself. Always compare against a human-labelled reference set when possible. Even a tiny validation table is better than judging a model from a few impressive examples.

**LLM connection:** fluent output can feel convincing even when it is wrong. Whether the system is a small classifier or a large language model, evaluation needs examples where you already know the expected answer. This is why accuracy checks, error review, and human-labelled reference data matter.

In [13]:
validation_records = [
    {"text": "Public access note says bus fare was too high for clinic visit", "human_label": "access_barrier"},
    {"text": "Rent arrears are causing stress at home", "human_label": "housing_stress"},
    {"text": "Public advice note says medication side effects need clinical advice", "human_label": "health_need"},
]

In [14]:
# Answer 7

### Question 8: Simple Evaluation

Compute a first check of model performance.

Guidance:
- count correct predictions with `.sum()`
- compute accuracy with `.mean()`
- print any incorrect rows

**Programming Plan**

- Boolean values can be counted: `True` behaves like `1`.
- Use `.sum()` for number correct.
- Use `.mean()` for accuracy.
- Filter rows where `correct` is `False` to inspect mistakes.

In [15]:
# Answer 8

## Part C: Summarise, Extract, and Check Failure Modes (Approx. 30 minutes)

**Working Pattern**

Summarisation and extraction are different tasks. A summary compresses text. Extraction turns text into fields. Failure-mode analysis asks when either process may mislead us. Keep these distinctions clear when designing research workflows.

**LLM connection:** many LLM applications are built from these same task types: summarise this document, extract these fields, classify this message, or explain why a case needs review. LLMs can do these tasks with more flexible language, but they can also omit details, invent unsupported details, or produce inconsistent structure. The simple functions here are deliberately limited so that the task boundaries are easy to see.

### Question 9: First-Sentence Summary

Write a simple summary function that returns the first sentence.

Guidance:
- split on `.`
- remove empty pieces
- add the full stop back

**Programming Plan**

- Split the long note on periods.
- Strip whitespace from each sentence.
- Ignore empty strings.
- Return the first sentence with a period added back.

In [16]:
long_note = (
    "A public engagement note reports missed appointments after a bus route changed. "
    "They report increased anxiety and difficulty paying rent. "
    "Clinic should offer phone appointment and signpost travel support."
)

In [17]:
# Answer 9

### Question 10: Keyword Summary

Return the first sentence containing a keyword.

Guidance:
- use `keywords`
- check each sentence in lower case
- fall back to `first_sentence_summary`

**Programming Plan**

- Reuse the sentence-splitting idea.
- Convert each sentence to lower case before checking keywords.
- Use a second `for` loop to check each keyword one at a time.
- Fall back to the first-sentence summary if no keyword appears.

In [18]:
keywords = ["rent", "urgent", "anxiety"]

In [19]:
# Answer 10

### Question 11: Extract Fields and Simple Entities

From each message, extract:
- area name
- money amounts such as `4 pounds`
- whether it mentions an appointment
- simple entity terms from `entity_terms`

**LLM connection:** extraction with an LLM often asks for structured output, such as JSON fields. The same design question appears here: decide the fields first, then check whether the extracted values are actually supported by the text.

Guidance:
- use string matching for areas and appointments
- use `re.findall` for money
- keep entity terms that appear in the lower-case text

**Programming Plan**

- For areas, loop through the known area names.
- For money, use `re.findall(...)`.
- For appointment, use a simple `"appointment" in text` test.
- For entities, keep terms that appear in the lower-case message.

In [20]:
known_areas = ["Northside", "Central", "Riverside"]
entity_terms = {
    "service": ["clinic", "hospital", "mental health", "medication"],
    "barrier": ["bus", "rent", "arrears", "internet"],
}

field_messages = [
    "Northside public respondent missed appointment after fare rose to 4 pounds.",
    "Central family reports rent arrears of 820 pounds.",
    "Riverside resident asks for mental health appointment.",
]

In [21]:
# Answer 11

### Optional Stretch: Mini Workflow and Failure Modes

This is an optional stretch task. If time is short, you can stop after Question 11 and use this section for extra practice or review.

Add summaries and entities to `validation_df`. Then run the model on two difficult examples and write one short review note for each.

Guidance:
- reuse `keyword_summary`
- reuse your entity extraction logic
- explain why each difficult example might need human review

**Programming Plan**

- Add new columns/fields one at a time.
- Reuse functions you already wrote.
- Print a compact table for inspection.
- For failure modes, explain the ambiguity in plain English.

**Research Note**

The goal is not to eliminate all errors in class. The goal is to notice where errors can enter: ambiguous wording, missing context, labels that overlap, and model confidence that may not match correctness.

**LLM connection:** this is the same reason many real LLM workflows keep a human-in-the-loop. The model can help sort, summarise, or draft, but a person often needs to review borderline cases, sensitive cases, and cases where the source text does not clearly support the output.

In [22]:
difficult_examples = [
    "Public respondent can attend clinic but is anxious about the rent letter",
    "No bus needed because appointment moved online, but portal failed",
]

In [23]:
# Optional Stretch Answer

## Stretch (Optional): Visualise Toy Attention Weights

This is not a transformer. It is only a small visual example of token weights. A text bar chart is enough for this lab.

**LLM connection:** transformer-based LLMs use attention mechanisms to combine information across tokens. Real attention is learned, layered, and much more complex than this toy list of weights. The point here is only to build an intuition that a model may give different parts of a text different influence.

In [24]:
toy_tokens = ["missed", "appointment", "because", "transport", "costs", "increased"]
attention_weights = [0.12, 0.18, 0.05, 0.30, 0.20, 0.15]

In [25]:
# Stretch Answer

## End of Lab Checklist

Before class ends, make sure you have:
- inspected tokenisation,
- run a small local classifier,
- compared model labels with human labels,
- extracted structured fields.

Optional stretch:
- combine summary, extraction, and classification in a mini workflow,
- name one failure mode.